# A4: how big is the domain shift, band by band

A2 showed the histograms and A3 the NDVI. A4 turns "these curves look different" into **numbers**, one per band and class, so the bands can be ranked and
the numbers can go into the report.

For every band (and NDVI), class and Thai product the Thai pixels are compared with the EuroSAT training pixels of the same class:

| measure | what it says | how to read it |
|---|---|---|
| **Shift in standard deviations** (`W1 / sd`) | the average distance between the two curves (the Wasserstein-1 or "earth mover's" distance), divided by the standard deviation of EuroSAT in the same class and band | 0 = the curves coincide; 1 = the Thai curve is on average one EuroSAT standard deviation away. Always positive |
| **Cohen's d** | difference of the means, divided by the pooled standard deviation | the sign tells the direction: positive = Thai is higher than EuroSAT, negative = lower. Around 0.2 is called small, 0.5 medium, 0.8 large |
| **Median difference** | Thai median minus EuroSAT median, in reflectance x 10000 (NDVI: in NDVI units) | the same direction, in the original unit |

The pixels are the same as in A2 and A3 (same seed, same order of drawing). The balanced mix takes the same number of pixels from forest, crop, built-up and water. Grassland is left out (no pure Thai patch).

**Keep in mind**
- Dividing by the EuroSAT standard deviation of the *same class* means a narrow class (water) shows large numbers for a small absolute change. The balanced mix is divided by the standard deviation of the mix, which also contains the differences between classes, so its numbers are smaller.
- The Thai images are a January-February median composite; every EuroSAT patch is a single date from any season. Part of every number comes from this, not from the region.
- Each Thai class comes from one or two study areas (see `DATA_NOTES.md`).
- These are descriptions of the pixels drawn, not statistical tests.

In [ ]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.append(str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

import config as c

# chart colours on a light surface (data-viz palette): a one-hue blue ramp for magnitude, blue-grey-red for direction
SURFACE, INK, INK2, MUTED, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#c3c2b7"
MAGNITUDE = LinearSegmentedColormap.from_list("magnitude", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
DIRECTION = LinearSegmentedColormap.from_list("direction", ["#2a78d6", "#f0efec", "#e34948"])

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.edgecolor": AXIS, "font.family": "sans-serif", "font.size": 10,
})

OUT = c.OUTPUT_DIR / "a4"
OUT.mkdir(parents=True, exist_ok=True)


def save(fig, name):
    # Windows sometimes refuses to open a file that was written a moment ago, so retry a few times
    path = OUT / name
    for attempt in range(5):
        try:
            fig.savefig(path, dpi=130, bbox_inches="tight")
            break
        except OSError:
            if attempt == 4:
                raise
            time.sleep(1)
    print("saved", path)

## 1. Load the data and draw the pixels (as in A2 and A3)

In [ ]:
EU_X = np.load(c.EUROSAT_DIR / "eurosat_X.npy", mmap_mode="r")
EU_Y = np.load(c.EUROSAT_DIR / "eurosat_y.npy")
EU_SPLIT = np.load(c.EUROSAT_DIR / "eurosat_split.npy")
TH_X = {"Thai L1C": np.load(c.PATCH_DIR / "thailand_X_L1C.npy", mmap_mode="r"),
        "Thai L2A": np.load(c.PATCH_DIR / "thailand_X_L2A.npy", mmap_mode="r")}
TH = pd.read_csv(c.PATCH_DIR / "thailand_patches.csv", keep_default_na=False)   # "nan" is an area name here

EVAL = c.THAI_EVAL_CLASSES      # forest, crop, built-up, water
PATCHES_PER_CLASS = 300
PIXELS_PER_CLASS = 100_000


def draw(patch_ids, rng):
    ids = np.sort(rng.choice(patch_ids, size=min(PATCHES_PER_CLASS, len(patch_ids)), replace=False))
    pix = np.sort(rng.choice(len(ids) * 64 * 64, size=min(PIXELS_PER_CLASS, len(ids) * 64 * 64), replace=False))
    return ids, pix


def take(X, ids, pix):
    block = np.asarray(X[ids])                                  # (patches, 12, 64, 64)
    return block.transpose(0, 2, 3, 1).reshape(-1, 12)[pix].astype("float32")


def with_ndvi(px):
    # the 12 bands plus NDVI as a 13th column (NaN where B8 + B4 is not positive)
    b8, b4 = px[:, c.BANDS.index("B8")], px[:, c.BANDS.index("B4")]
    total = b8 + b4
    nd = np.where(total > 0, (b8 - b4) / np.where(total > 0, total, 1), np.nan)
    return np.column_stack([px, nd])


rng = np.random.default_rng(c.SEED)
samples = {"EuroSAT (train)": {}, "Thai L1C": {}, "Thai L2A": {}}
for cls in EVAL:
    eu_ids = np.flatnonzero((EU_Y == c.CLASSES.index(cls)) & (EU_SPLIT == 0))
    ids, pix = draw(eu_ids, rng)
    samples["EuroSAT (train)"][cls] = with_ndvi(take(EU_X, ids, pix))
    th_ids = TH.loc[TH["pure"] & (TH["label"] == cls), "patch_id"].to_numpy()
    ids, pix = draw(th_ids, rng)
    for name, X in TH_X.items():
        samples[name][cls] = with_ndvi(take(X, ids, pix))

for name in samples:
    samples[name]["balanced mix"] = np.concatenate([samples[name][cls] for cls in EVAL])
GROUPS = EVAL + ["balanced mix"]
FEATURES = c.BANDS + ["NDVI"]
print({name: samples[name]["balanced mix"].shape for name in samples})

## 2. Measure the shift

For every product (Thai L1C, Thai L2A), group and band: the three measures above. The result is saved as `outputs/a4/band_shift.csv`.

In [ ]:
PROBS = (np.arange(2000) + 0.5) / 2000


def shift(e, t):
    # e: EuroSAT values, t: Thai values (any lengths); NaN is dropped
    e, t = e[np.isfinite(e)], t[np.isfinite(t)]
    qe, qt = np.quantile(e, PROBS), np.quantile(t, PROBS)
    w1 = np.mean(np.abs(qt - qe))                                  # Wasserstein-1 distance
    d = (t.mean() - e.mean()) / np.sqrt((t.var() + e.var()) / 2)   # Cohen's d
    return {"W1_over_sd": w1 / e.std(), "cohen_d": d, "median_diff": np.median(t) - np.median(e)}


records = []
for product in ("Thai L1C", "Thai L2A"):
    for group in GROUPS:
        for k, feature in enumerate(FEATURES):
            m = shift(samples["EuroSAT (train)"][group][:, k], samples[product][group][:, k])
            records.append({"product": product, "group": group, "feature": feature, **m})
shifts = pd.DataFrame(records)
shifts.round(3).to_csv(OUT / "band_shift.csv", index=False)
print(len(shifts), "rows")

## 3. Heatmap: how far is Thailand from EuroSAT? (size of the shift)

Every cell is the shift in EuroSAT standard deviations (`W1 / sd`) for one band and one class. A darker cell means a bigger shift. The colour scale stops at 2; the numbers are the real values.

In [ ]:
def heatmap(ax, matrix, rows, cols, cmap, vmin, vmax, fmt, title):
    ax.imshow(matrix, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(cols)))
    ax.set_xticklabels(cols, rotation=20, ha="right")
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels(rows)
    ax.set_xticks(np.arange(-0.5, len(cols)), minor=True)
    ax.set_yticks(np.arange(-0.5, len(rows)), minor=True)
    ax.grid(which="minor", color=SURFACE, linewidth=2)
    ax.tick_params(which="both", length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(title, loc="left", fontsize=12, color=INK)
    norm = plt.Normalize(vmin, vmax)
    for i in range(len(rows)):
        for j in range(len(cols)):
            r, g, b, _ = cmap(norm(np.clip(matrix[i, j], vmin, vmax)))
            ax.text(j, i, format(matrix[i, j], fmt), ha="center", va="center", fontsize=8.5,
                    color="white" if 0.299 * r + 0.587 * g + 0.114 * b < 0.55 else INK)


def two_panels(column, cmap, vmin, vmax, fmt, label, title, filename):
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 6.4), sharey=True)
    for ax, product in zip(axes, ("Thai L1C", "Thai L2A")):
        table = shifts[shifts["product"] == product].pivot(index="feature", columns="group", values=column).reindex(index=FEATURES, columns=GROUPS)
        heatmap(ax, table.to_numpy(), FEATURES, GROUPS, cmap, vmin, vmax, fmt, f"{product} against EuroSAT")
    bar = fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(vmin, vmax), cmap=cmap), ax=axes, fraction=0.025, pad=0.02)
    bar.set_label(label, color=INK2)
    bar.outline.set_visible(False)
    fig.suptitle(title, y=1.0, fontsize=13, color=INK)
    save(fig, filename)
    plt.show()


two_panels("W1_over_sd", MAGNITUDE, 0, 2, ".2f", "shift in EuroSAT standard deviations",
           "Size of the domain shift per band and class (darker = bigger shift)", "shift_size.png")

## 4. Heatmap: in which direction? (Cohen's d)

Red: Thai pixels are higher than EuroSAT in that band; blue: lower. Grey is no difference. The colour scale stops at +/-1.5.

In [ ]:
two_panels("cohen_d", DIRECTION, -1.5, 1.5, ".2f", "Cohen's d  (red: Thai higher, blue: Thai lower)",
           "Direction of the domain shift per band and class", "shift_direction.png")

## 5. Which bands shift most?

The three bands (NDVI included) with the biggest shift in EuroSAT standard deviations, for every group and product:

In [ ]:
rows = []
for product in ("Thai L1C", "Thai L2A"):
    for group in GROUPS:
        part = shifts[(shifts["product"] == product) & (shifts["group"] == group)].sort_values("W1_over_sd", ascending=False).head(3)
        rows.append({"product": product, "group": group,
                     "three biggest shifts": ", ".join(f"{r.feature} ({r.W1_over_sd:.2f})" for r in part.itertuples())})
pd.DataFrame(rows).set_index(["product", "group"])

And the mean shift over the 12 bands (NDVI not counted) for every group: one number per class that says how far Thailand is from EuroSAT in total.

In [ ]:
bands_only = shifts[shifts["feature"].isin(c.BANDS)]
overall = bands_only.groupby(["group", "product"])["W1_over_sd"].mean().unstack().reindex(GROUPS)
overall.round(2)

## 6. Questions to answer for the slide

- Which bands shift most, and is that the same in L1C and L2A? (Watch B9 in L2A: it is very different there, so do not read it as a real land difference before it is checked.)
- Which class is the furthest from its EuroSAT version, and which is the closest?
- Is the direction the same for all classes? If crop goes the other way from forest and built-up, one correction for the whole country cannot fix it.
- Which part of the shift could come from the January-February composite and not from the region?